# Semi-coherent likelihood — utility functions (proof-of-concept)

This notebook forms the basis of [`semi-coherent-likelihood.md`](semi-coherent-likelihood.md): a single differentiable scalar field on the EMRI parameter space

$$\Lambda(\vec\theta),\qquad \vec\theta=(M,\mu,a,T_{\rm plunge},e_f),$$

obtained by chaining a `fewtrax` backward trajectory into the single-harmonic semi-coherent statistic `emrisearch.jax_utils.det_stat`, and the tooling needed to  characterise its geometry (gradient checks, Fisher/metric, a multimodal JAX sampler, mode clustering, non-Gaussianity diagnostics).


* The likelihood is **frequency-track only** (no amplitude/response modelling): how much can we constrain from the track alone?question.
* `logL(theta) = LOGL_FACTOR * det_stat(...)` with `LOGL_FACTOR = 0.5`, following the note
  ($\log\mathcal L=\tfrac12\Lambda$ up to a $\theta$-independent constant). The factor
  is exposed because it sets the effective sampling temperature.
* The sampler is **blackjax** (NUTS for within-mode characterisation + tempered SMC /
  many-chain global runs for the secondary-mode census). PARIS is *not* used here; the
  physics study is the point, not the sampler. `flowMC` is noted as a drop-in alternative
  for genuinely multimodal surfaces.
* **Stage 1 (this notebook):** the "data" comes from a known **reference system** — we
  expose both (a) the reference `(f, fdot)` track and (b) injected `data_sfts`.

Run on GPU: everything is `jit`/`vmap`-able and uses `float64` (enabled below).


In [1]:
# --- Environment ----------------------------------------------------------
import os
# Recommended for many small batched solves on GPU (uncomment if memory allows):
# os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.9")

import jax
jax.config.update("jax_enable_x64", True)   # MUST precede heavy jax use
import jax.numpy as jnp
import numpy as np
import diffrax

print("jax", jax.__version__, "| devices:", jax.devices())

# Core building blocks already in the repos -------------------------------
from emrisearch.jax_utils import det_stat, psd          # differentiable Lambda + PSD
from fewtrax.data import load_flux_data
from fewtrax.trajectory.inspiral import (
    EMRIInspiral, SEPARATRIX_BUFFER, _x_sign,
)
from fewtrax.utils.geodesic import get_separatrix_fast
from fewtrax.trajectory.helpers import dense_phase_derivs
from fewtrax.utils.constants import MTSUN_SI, YEAR_SI

try:
    import blackjax
    print("blackjax", blackjax.__version__)
except Exception as e:
    print("blackjax not importable yet:", e)


jax 0.7.2 | devices: [CudaDevice(id=0)]
Using TDI PSD
blackjax not importable yet: No module named 'blackjax'


## 1. Configuration

A single fixed reference $(M,\mu,a)$ as in the note; only $(e_f,\text{SNR},\text{seed})$
are scanned in the week-2 campaign. The default reference matches the smoke-test values
used elsewhere in the repo (`true_values = [1e6, 10, 0.1, Tpl, e_f, x0]`).

Note the **eccentricity bound is widened to 0.7** (the study control variable);
`DEFAULT_BOUNDS` in `track_optimizer.py` caps $e_f$ at 0.3.

In [2]:
# --- Reference system (the 'data' source for stage 1) --------------------
M_REF      = 1.0e6    # primary mass   [Msun]
MU_REF     = 10.0     # secondary mass [Msun]
A_REF      = 0.1      # dimensionless spin
TPL_REF    = 0.2      # time from data start to plunge [yr]
EF_REF     = 0.1      # eccentricity at plunge
X0         = 1.0      # prograde equatorial

# --- Data / SFT settings -------------------------------------------------
T_DATA   = 0.2        # observation window [yr]
T_SNR    = 0.2        # duration over which SNR is defined (>= T_DATA)
DELTA_T  = 5.0        # waveform sampling [s]
T_SFT    = 5.0e4      # SFT segment length [s]  (det_stat default)
SNR_REF  = 30.0       # target matched-filter SNR of the injection
P_BINS   = 100        # +/- bins around the central SFT bin in det_stat
MODE     = (2, 0)     # dominant harmonic (m, n): f_alpha = m f_phi + n f_r

LOGL_FACTOR = 0.5     # logL = LOGL_FACTOR * det_stat(...)  (sets temperature)

# --- Parameter box  [M, mu, a, T_plunge, e_f] ----------------------------
# (e_f widened to 0.7 for the eccentricity study)
STUDY_BOUNDS = np.array([
    [3.0e5, 5.0e6],   # M
    [1.0,   100.0],   # mu
    [0.0,   0.998],   # a
    [0.05,  2.0],     # T_plunge [yr]
    [0.0,   0.7],     # e_f
])
PARAM_NAMES = ["M", "mu", "a", "T_plunge", "e_f"]

THETA_REF = np.array([M_REF, MU_REF, A_REF, TPL_REF, EF_REF])
print("theta_ref =", dict(zip(PARAM_NAMES, THETA_REF)))


theta_ref = {'M': np.float64(1000000.0), 'mu': np.float64(10.0), 'a': np.float64(0.1), 'T_plunge': np.float64(0.2), 'e_f': np.float64(0.1)}


In [4]:
# --- Flux data (static; one instance is reused everywhere) ---------------
# Point load_flux_data at your FEW data dir if it is not auto-discovered:
#   flux_data = load_flux_data("/path/to/few/data")
from dotenv import load_dotenv
load_dotenv()
DATA_DIR = os.getenv("FEW_DATA_DIR")
print(f"FEW_DATA_DIR = {DATA_DIR}")

flux_data = load_flux_data()
print("flux data loaded")


FEW_DATA_DIR = /data/leuven/367/vsc36785/LISA/FastEMRIWaveforms/data
flux data loaded


## 2. Parameter transforms (constrained $\leftrightarrow$ unconstrained)

We sample in the **unconstrained** coordinates already used by `TrackOptimizerJAX`:
$\theta_{\rm phys}= \ell + (u-\ell)\,\sigma(\theta_{\rm raw})$, with $\sigma$ the logistic.
For a **uniform prior on the box**, the target density in raw space picks up the
log-Jacobian of the sigmoid map (`log_det_jac`), which `log_prior_raw` includes.

In [5]:
LO = jnp.asarray(STUDY_BOUNDS[:, 0])
HI = jnp.asarray(STUDY_BOUNDS[:, 1])

def to_phys(raw):
    '''Unconstrained R^5 -> physical box via the logistic map.'''
    return LO + (HI - LO) * jax.nn.sigmoid(raw)

def to_raw(phys):
    '''Physical box -> unconstrained R^5 (inverse logit). Accepts np or jnp.'''
    phys = jnp.asarray(phys)
    u = jnp.clip((phys - LO) / (HI - LO), 1e-6, 1.0 - 1e-6)
    return jnp.log(u / (1.0 - u))

def log_det_jac(raw):
    '''log |d theta_phys / d theta_raw| for the sigmoid map (sum over dims).'''
    # d/dx [lo+(hi-lo) sigmoid(x)] = (hi-lo) sigmoid(x)(1-sigmoid(x))
    log_s = jax.nn.log_sigmoid(raw) + jax.nn.log_sigmoid(-raw)   # log[sig(1-sig)]
    return jnp.sum(jnp.log(HI - LO) + log_s)

def log_prior_raw(raw):
    '''Uniform prior on the box, expressed in raw coordinates.'''
    # Uniform density on the box is constant in phys space; the raw-space
    # density is that constant times the Jacobian. The additive log|box| const
    # is dropped (irrelevant for sampling).
    return log_det_jac(raw)

THETA_REF_RAW = to_raw(THETA_REF)
print("round-trip check:", np.allclose(np.asarray(to_phys(THETA_REF_RAW)), THETA_REF, rtol=1e-5))


round-trip check: True


## 3. Differentiable frequency track $\theta \to (f_\alpha, \dot f_\alpha)$

A thin re-implementation of `EMRIInspiral.get_f_fdot_fddot_back` that

* uses the **`TrackOptimizerJAX` plunge convention** $\tau = T_{\rm plunge}\!\cdot\!{\rm yr} - t_\alpha$
  (so $T_{\rm plunge}$ is a genuine free parameter measured from the data start), and
* takes a **selectable diffrax adjoint** so the same code serves `grad`/`jacrev`
  (default `RecursiveCheckpointAdjoint`) and the forward-mode paths
  (`DirectAdjoint`, needed for `jacfwd`).

Both fundamental tracks ($\Phi_\phi$ and $\Phi_r$) are returned; a harmonic
$(m,n)$ is the integer combination $m f_\phi + n f_r$.

In [6]:
def make_track_fn(flux_data, t_alpha, adjoint="recursive",
                  x0=X0, max_steps=512, atol=1e-10, rtol=1e-10):
    '''Return track(M, mu, a, T_plunge, e_f) -> ((f_phi,fdot_phi,fddot_phi),
    (f_r,fdot_r,fddot_r)) evaluated on the SFT mid-times t_alpha [s].

    adjoint: 'recursive' (grad/jacrev) or 'direct' (jacfwd/forward-mode).
    '''
    adj = diffrax.DirectAdjoint() if adjoint == "direct" else diffrax.RecursiveCheckpointAdjoint()
    inst = EMRIInspiral(flux_data, adjoint=adj)
    t_alpha_ = jnp.asarray(t_alpha, dtype=jnp.float64)

    def track(M, mu, a, T_plunge, e_f):
        a_   = jnp.asarray(a, jnp.float64)
        e_f_ = jnp.asarray(e_f, jnp.float64)
        M_s  = (M + mu) * MTSUN_SI
        x_in = _x_sign(a_, x0)

        # backward solve spans tau in [0 (plunge), T_plunge*yr / M_s (data start)]
        T_geo     = (T_plunge * YEAR_SI) / M_s
        mu_over_M = M * mu / (M + mu) ** 2
        r_isco    = get_separatrix_fast(jnp.abs(a_), jnp.zeros((), jnp.float64), x_in)
        ode_args  = (mu_over_M, a_, x0, r_isco)

        p_sep = get_separatrix_fast(jnp.abs(a_), e_f_, x_in)
        y0    = jnp.array([p_sep + SEPARATRIX_BUFFER, e_f_, 0.0, 0.0, 0.0], jnp.float64)

        sol = diffrax.diffeqsolve(
            diffrax.ODETerm(lambda t, y, args: -inst._ode_rhs(t, y, args)),
            diffrax.Dopri8(),
            t0=jnp.zeros((), jnp.float64), t1=T_geo, dt0=None, y0=y0,
            saveat=diffrax.SaveAt(dense=True),
            stepsize_controller=diffrax.PIDController(rtol=rtol, atol=atol),
            max_steps=max_steps, args=ode_args, adjoint=inst.adjoint, throw=False,
        )

        # plunge at tau=0; clamp into [0, T_geo] so bins after the (sampled)
        # plunge reuse the plunge-edge value instead of extrapolating the
        # degree-6 dense interpolant to tau<0 (matches jnp.interp edge behavior).
        tau_query = jnp.clip((T_plunge * YEAR_SI - t_alpha_) / M_s, 0.0, T_geo)
        interp    = sol.interpolation
        two_pi_Ms = 2.0 * jnp.pi * M_s

        def freqs(tau, comp):
            d1, d2, d3 = dense_phase_derivs(interp, tau, comp)
            return (-d1 / two_pi_Ms,             # f      [Hz]
                    d2 / (two_pi_Ms * M_s),       # fdot   [Hz/s]
                    -d3 / (two_pi_Ms * M_s**2))   # fddot  [Hz/s^2]

        f_phi, fdot_phi, fddot_phi = jax.vmap(lambda t: freqs(t, 2))(tau_query)
        f_r,   fdot_r,   fddot_r   = jax.vmap(lambda t: freqs(t, 4))(tau_query)
        return (f_phi, fdot_phi, fddot_phi), (f_r, fdot_r, fddot_r)

    return track


def f_fdot_of_theta(track, theta_phys, mode):
    '''(f_alpha, fdot_alpha) for harmonic (m, n) from a track fn.'''
    M, mu, a, T_plunge, e_f = (theta_phys[0], theta_phys[1], theta_phys[2],
                               theta_phys[3], theta_phys[4])
    (f_phi, fdot_phi, _), (f_r, fdot_r, _) = track(M, mu, a, T_plunge, e_f)
    m, n = mode
    return m * f_phi + n * f_r, m * fdot_phi + n * fdot_r


## 4. Stage-1 data

### 4a. Reference `(f, fdot)` track — the literal "data from a reference system"

Computed from the differentiable model at `THETA_REF`. This is the cheap object the
first proof-of-concept likelihood (the track-residual likelihood, §5a) compares against.

In [7]:
# SFT mid-time grid for the chosen data window
N_SFT     = int(np.floor(T_DATA * YEAR_SI / T_SFT))
T_ALPHA   = (np.arange(N_SFT) + 0.5) * T_SFT          # mid-times [s]
print(f"N_SFT = {N_SFT}, t_alpha in [{T_ALPHA[0]:.0f}, {T_ALPHA[-1]:.0f}] s")

track_grad   = make_track_fn(flux_data, T_ALPHA, adjoint="recursive")
track_direct = make_track_fn(flux_data, T_ALPHA, adjoint="direct")

# Reference track = the 'observed' (f, fdot) pairs
f_data, fdot_data = f_fdot_of_theta(track_grad, jnp.asarray(THETA_REF), MODE)
f_data, fdot_data = np.asarray(f_data), np.asarray(fdot_data)
print("f_data    [Hz]   :", f_data[:3], "...", f_data[-3:])
print("fdot_data [Hz/s] :", fdot_data[:3], "...", fdot_data[-3:])


N_SFT = 126, t_alpha in [25000, 6275000] s
f_data    [Hz]   : [0.0031316  0.00313562 0.00313967] ... [0.00433526 0.0043975  0.00449078]
fdot_data [Hz/s] : [8.00933064e-11 8.06326527e-11 8.11794124e-11] ... [1.08790529e-09 1.44776768e-09 2.55602855e-09]


### 4b. Injected `data_sfts` — the semi-coherent likelihood data

`generate_emri_signal_and_sfts` builds an EMRI signal with FEW, rescales the distance
to hit `SNR_REF`, adds a noise realisation and returns the SFTs. The model `T_plunge`
that reproduces this injection's track is `TPL_REF` (plunge measured from the data start),
consistent with the convention in §3.

> Requires `few` (NumPy backend). One injection = one grid cell; the week-2 loop simply
> varies `EF_REF`, `SNR_REF` and the noise seed.

In [8]:
from emrisearch.search_utils import generate_emri_signal_and_sfts

def make_injection(M, mu, a, T_plunge, e_f, snr, seed=0, zero_noise=False):
    '''Return (data_sfts [complex128, (Nf, Nsft)], t_alpha [s], info dict).

    Set zero_noise=True for the noiseless reference run (isolates likelihood
    structure from noise scatter, as in the week-2 grid).
    '''
    np.random.seed(seed)
    true_values = np.array([M, mu, a, T_plunge, e_f, X0])
    inj = generate_emri_signal_and_sfts(true_values, T_DATA, T_SFT, DELTA_T, snr, T_SNR)
    sfts = inj["signal_sfts"] if zero_noise else inj["data_sfts"]
    data_sfts = jnp.asarray(sfts, dtype=jnp.complex128)
    return data_sfts, np.asarray(inj["t_alpha"]), inj

data_sfts, t_alpha_inj, inj = make_injection(
    M_REF, MU_REF, A_REF, TPL_REF, EF_REF, SNR_REF, seed=0)
print("data_sfts:", data_sfts.shape, "| snr_final:", inj.get("snr_final"))

# det_stat needs len(f_alpha) == data_sfts.shape[1]; rebuild track fn on this grid.
track_grad   = make_track_fn(flux_data, t_alpha_inj, adjoint="recursive")
track_direct = make_track_fn(flux_data, t_alpha_inj, adjoint="direct")


Using TDI PSD
Using response generator
Waveform generated on CPU
Final SNR: 29.999999999999996
data_sfts: (5001, 126) | snr_final: 29.999999999999996


## 5. The likelihoods (functions of the raw, unconstrained $\theta$)

* **5a. Track-residual** (`logL_track`): a fast Gaussian on the `(f, fdot)` residuals
  vs the reference track. No SFTs, trivial to differentiate — ideal for first
  gradient checks and sampler smoke tests (the literal "f, fdot pairs" stage).
* **5b. Semi-coherent** (`logL_semicoh`): the real study object,
  `LOGL_FACTOR * det_stat(data_sfts, f_alpha(theta), fdot_alpha(theta))`.

Each is wrapped as `log_posterior_*` = `logL` + `log_prior_raw` for the sampler.

In [9]:
# --- 5a. Track-residual likelihood (proof of concept & stage-2 interface) ---
# The search pipeline (TrackOptimizerJAX / iterative_track_search) both consumes
# and emits a per-segment {f, fdot} SEQUENCE on the SFT bins t_alpha, shape (Nsft,)
# (see iterative_track_search._frequency_and_fdot_at_obs). This factory builds a
# Gaussian likelihood comparing the MODEL track to such an observed {f, fdot}
# sequence. Stage 1 feeds it the reference track; stage 2 feeds it the search
# pipeline output -- identical shape, identical code.

def make_logL_track(f_seq, fdot_seq, track=None, mode=MODE,
                    sigma_f=1.0 / T_SFT, sigma_fdot=1.0 / T_SFT**2,
                    use_fdot=True):
    """Gaussian logL on the {f, fdot} residuals vs an observed sequence.

    f_seq, fdot_seq : (Nsft,) observed track on t_alpha (reference, or the
                      search-statistic output for stage 2).
    sigma_f, sigma_fdot : per-bin uncertainties (default: SFT resolution scales).
    use_fdot : include the fdot residual term (set False to use f only).
    """
    track    = track if track is not None else track_grad
    f_obs    = jnp.asarray(f_seq)
    fdot_obs = jnp.asarray(fdot_seq)
    valid    = jnp.asarray(np.asarray(f_seq) > 0)
    def logL(raw):
        theta = to_phys(raw)
        f_p, fdot_p = f_fdot_of_theta(track, theta, mode)
        r_f = jnp.where(valid, (f_p - f_obs) / sigma_f, 0.0)
        out = jnp.sum(r_f**2)
        if use_fdot:
            r_fd = jnp.where(valid, (fdot_p - fdot_obs) / sigma_fdot, 0.0)
            out = out + jnp.sum(r_fd**2)
        return -0.5 * out
    return logL

# Stage-1 instance: data = reference {f, fdot} track from THETA_REF
logL_track = make_logL_track(f_data, fdot_data)

# --- 5b. Semi-coherent likelihood (the real object) ----------------------
def make_logL_semicoh(track, data_sfts, mode=MODE, factor=LOGL_FACTOR,
                      P=P_BINS, T_sft=T_SFT):
    def logL(raw):
        theta = to_phys(raw)
        f_a, fdot_a = f_fdot_of_theta(track, theta, mode)
        return factor * det_stat(data_sfts, f_a, fdot_a, P=P, T_sft=T_sft)
    return logL

logL_semicoh = make_logL_semicoh(track_grad, data_sfts)

def make_log_posterior(logL_fn):
    def log_posterior(raw):
        return logL_fn(raw) + log_prior_raw(raw)
    return log_posterior

log_post_track   = make_log_posterior(logL_track)
log_post_semicoh = make_log_posterior(logL_semicoh)

print("logL_track(ref)   =", float(logL_track(THETA_REF_RAW)))
print("logL_semicoh(ref) =", float(logL_semicoh(THETA_REF_RAW)))


logL_track(ref)   = -13.137291443791097
logL_semicoh(ref) = 332.81710167512597


## 6. Gradient check vs finite differences (week-1 deliverable 1)

Validate $\nabla_{\vec\theta}\Lambda$ against central differences. We check in **raw**
coordinates (what the sampler sees). Finite differences are taken in physical space and
mapped through the analytic Jacobian, or directly in raw space — here we do raw space
directly for an apples-to-apples comparison with `jax.grad`.

In [10]:
def fd_grad(f, x, rel=1e-4, abs_floor=1e-3):
    x = np.asarray(x, dtype=float)
    g = np.zeros_like(x)
    for i in range(x.size):
        h = rel * max(abs(x[i]), abs_floor)
        xp, xm = x.copy(), x.copy()
        xp[i] += h; xm[i] -= h
        g[i] = (float(f(jnp.asarray(xp))) - float(f(jnp.asarray(xm)))) / (2 * h)
    return g

for name, fn in [("track", logL_track), ("semicoh", logL_semicoh)]:
    g_ad = np.asarray(jax.grad(fn)(THETA_REF_RAW))
    g_fd = fd_grad(fn, THETA_REF_RAW)
    denom = np.maximum(np.abs(g_ad) + np.abs(g_fd), 1e-30)
    rel_err = np.abs(g_ad - g_fd) / denom
    print(f"[{name}] max rel err = {rel_err.max():.2e}")
    print("   ad:", np.array2string(g_ad, precision=3))
    print("   fd:", np.array2string(g_fd, precision=3))


[track] max rel err = 6.99e-05
   ad: [-3303.78   -456.167   396.142 -4091.102  -146.42 ]
   fd: [-3303.734  -456.177   396.156 -4090.86   -146.4  ]
[semicoh] max rel err = 3.29e-02
   ad: [5611.347 1255.09  -662.614 2250.994  282.554]
   fd: [5765.173 1279.504 -707.657 2380.597  282.617]


## 7. `jit` + `vmap` validation

Confirm the jitted, batched evaluation reproduces the serial result, both over a batch
of $\theta$ and over the harmonic mode. This is what the week-2 campaign relies on to
launch the grid in parallel on GPU.

In [ ]:
logL_jit = jax.jit(logL_semicoh)
print("jit matches:", np.allclose(float(logL_jit(THETA_REF_RAW)),
                                  float(logL_semicoh(THETA_REF_RAW))))

# Batch over theta (e.g. a slice through the box)
key = jax.random.PRNGKey(0)
batch_raw = THETA_REF_RAW[None, :] + 0.1 * jax.random.normal(key, (8, 5))
vals_v = jax.vmap(logL_jit)(batch_raw)
vals_s = jnp.array([logL_semicoh(r) for r in batch_raw])
print("vmap-over-theta matches:", np.allclose(np.asarray(vals_v), np.asarray(vals_s)))

# vmap over harmonic mode at fixed theta
def logL_mode(raw, m, n):
    theta = to_phys(raw)
    f_a, fdot_a = f_fdot_of_theta(track_grad, theta, (m, n))
    return LOGL_FACTOR * det_stat(data_sfts, f_a, fdot_a, P=P_BINS, T_sft=T_SFT)

ms = jnp.array([2, 2, 3, 4]); ns = jnp.array([0, 1, 0, -1])
vals_modes = jax.vmap(lambda m, n: logL_mode(THETA_REF_RAW, m, n))(ms, ns)
print("Lambda over modes (2,0),(2,1),(3,0),(4,-1):", np.asarray(vals_modes))


jit matches: True


The kernel dies here. Maybe an issue with installing blackjax, which I was doing at the same moment. 

## 8. Curvature / metric tooling (week-1 deliverable 3)

The note wants the **observed Fisher** $F_{ij}=-\partial_i\partial_j\log\mathcal L$ at the
MAP, its eigen-decomposition, the Cramér–Rao covariance $\Sigma=F^{-1}$, and a
`mismatch` helper for the template-density study (D).

**Important numerical note.** `det_stat`'s Fresnel kernel is a `custom_vjp` with **no
`jvp` rule**, so `jax.hessian` (forward-over-reverse) on the semi-coherent logL is
unreliable. We therefore provide:

1. `fisher_fd` — a **finite-difference Hessian** of the scalar logL (robust, faithful to
   the semi-coherent object; the primary tool for studies B/C/D).
2. `fisher_gauss_newton` — the **$J^\top J/\sigma_f^2$ track Fisher** (first-order
   Jacobian only; cheap, exact for the linear-signal approximation; good cross-check and
   the natural object for the metric in study D).

In [ ]:
def fisher_fd(logL_fn, raw0, rel=1e-3, abs_floor=1e-2):
    '''Observed Fisher F = -Hessian(logL) by central finite differences (raw coords).'''
    x0 = np.asarray(raw0, dtype=float); n = x0.size
    h = np.maximum(np.abs(x0), abs_floor) * rel
    f = lambda x: float(logL_fn(jnp.asarray(x)))
    f0 = f(x0)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            if i == j:
                xp, xm = x0.copy(), x0.copy()
                xp[i] += h[i]; xm[i] -= h[i]
                H[i, i] = (f(xp) - 2 * f0 + f(xm)) / h[i]**2
            else:
                xpp, xpm, xmp, xmm = (x0.copy() for _ in range(4))
                xpp[i] += h[i]; xpp[j] += h[j]
                xpm[i] += h[i]; xpm[j] -= h[j]
                xmp[i] -= h[i]; xmp[j] += h[j]
                xmm[i] -= h[i]; xmm[j] -= h[j]
                H[i, j] = H[j, i] = (f(xpp) - f(xpm) - f(xmp) + f(xmm)) / (4 * h[i] * h[j])
    return -H   # Fisher = -Hessian(logL)


def fisher_gauss_newton(track, theta_phys, mode=MODE, sigma_f=None):
    '''Gauss-Newton track Fisher F_ij = (J^T J)/sigma_f^2 in PHYSICAL coords.

    J_ai = d f_pred(t_a) / d theta_i  (only the f track; add fdot similarly if wanted).
    '''
    if sigma_f is None:
        sigma_f = 1.0 / T_SFT
    def predict(theta):
        f_a, _ = f_fdot_of_theta(track, theta, mode)
        return f_a
    J = np.asarray(jax.jacobian(predict)(jnp.asarray(theta_phys)))   # (Nsft, 5)
    return (J.T @ J) / sigma_f**2


def metric_report(F, names=PARAM_NAMES):
    '''Eigen-structure, Cramer-Rao sigmas and condition number of a Fisher matrix.'''
    F = np.asarray(F)
    w, V = np.linalg.eigh(F)
    cov = np.linalg.pinv(F)
    sig = np.sqrt(np.clip(np.diag(cov), 0, None))
    return {
        "eigvals": w,
        "eigvecs": V,                      # columns; smallest eigval = worst-constrained dir
        "cov": cov,
        "sigma_marginal": dict(zip(names, sig)),
        "cond": (w.max() / max(w.min(), 1e-300)),
        "sqrt_det_g": float(np.sqrt(np.clip(np.linalg.det(F), 0, None))),
    }


def mismatch(F, dtheta):
    '''Quadratic mismatch ~ 0.5 dtheta^T F dtheta (study D template density).'''
    d = np.asarray(dtheta)
    return float(0.5 * d @ np.asarray(F) @ d)


In [ ]:
# Demonstrate at the reference (treated as the MAP here; in production use the
# per-run MAP from the sampler).
F_fd = fisher_fd(logL_semicoh, THETA_REF_RAW)              # raw-coord observed Fisher
rep_fd = metric_report(F_fd, names=[f"raw_{n}" for n in PARAM_NAMES])
print("FD observed Fisher (raw coords):")
print("  eigvals:", np.array2string(rep_fd["eigvals"], precision=3))
print("  cond   :", f"{rep_fd['cond']:.2e}")

F_gn = fisher_gauss_newton(track_grad, THETA_REF, MODE)    # physical-coord track Fisher
rep_gn = metric_report(F_gn)
print("\nGauss-Newton track Fisher (physical coords):")
print("  marginal sigmas:", {k: f'{v:.3e}' for k, v in rep_gn['sigma_marginal'].items()})
print("  worst-constrained eigvec (expect ~ M-mu ridge):",
      np.array2string(rep_gn['eigvecs'][:, 0], precision=3))


## 9. Sampler — blackjax

Two complementary modes, both `vmap`-friendly on GPU:

* **9a. NUTS, many parallel chains** with window adaptation — characterises the dominant
  mode (study B) and, when seeded from dispersed points, also discovers secondaries
  (a robust, version-stable route to study A via clustering, §10).
* **9b. Adaptive tempered SMC** — a principled global sampler for the multimodal surface
  (study A). The blackjax SMC API varies across versions; the cell is written against a
  recent release and flagged accordingly.

> `flowMC` (normalizing-flow + MALA/HMC, as used in `jim`) is a strong drop-in
> alternative for this multimodal problem — swap it in if NUTS/SMC mixing is poor.

In [ ]:
# --- 9a. NUTS with window adaptation, vmapped over chains ----------------
def build_nuts_runner(log_density_fn):
    def run_chain(key, init_pos, n_warmup, n_samples):
        k_warm, k_sample = jax.random.split(key)
        warmup = blackjax.window_adaptation(blackjax.nuts, log_density_fn)
        (state, params), _ = warmup.run(k_warm, init_pos, num_steps=n_warmup)
        step = blackjax.nuts(log_density_fn, **params).step
        def one(state, k):
            state, info = step(k, state)
            return state, (state.position, info.acceptance_rate)
        keys = jax.random.split(k_sample, n_samples)
        _, (positions, acc) = jax.lax.scan(one, state, keys)
        return positions, acc
    return run_chain


def run_nuts_multichain(log_density_fn, init_positions, n_warmup=500,
                        n_samples=1000, seed=0):
    '''init_positions: (n_chains, 5) in raw coords. Returns (n_chains, n_samples, 5).'''
    runner = build_nuts_runner(log_density_fn)
    keys = jax.random.split(jax.random.PRNGKey(seed), init_positions.shape[0])
    batched = jax.vmap(runner, in_axes=(0, 0, None, None))
    positions, acc = batched(keys, init_positions, n_warmup, n_samples)
    return positions, acc


# Dispersed initial points (broad over the box) for global coverage
def disperse_init(n_chains, seed=0, scale=1.0):
    k = jax.random.PRNGKey(seed)
    # uniform in phys box -> raw, plus jitter
    u = jax.random.uniform(k, (n_chains, 5))
    phys = LO + (HI - LO) * (0.05 + 0.9 * u)        # stay off the hard edges
    return jax.vmap(to_raw)(phys) * scale

# Example (uncomment to run on GPU):
# init = disperse_init(16, seed=1)
# samples, acc = run_nuts_multichain(log_post_semicoh, init, n_warmup=500, n_samples=1500)
# print("samples:", samples.shape, "mean accept:", float(acc.mean()))


In [ ]:
# --- 9b. Adaptive tempered SMC (global, multimodal) ----------------------
# API note: validated structure for blackjax >= 1.0. If your version differs,
# the moving parts are: (loglikelihood, logprior, an MCMC mutation kernel,
# a resampling fn, and a target-ESS schedule).
from blackjax.smc import resampling

def run_tempered_smc(loglik_fn, logprior_fn, init_particles, seed=0,
                     n_mcmc_steps=20, target_ess=0.6, step_size=1e-2):
    '''init_particles: (n_particles, 5) raw coords. Returns final particles + n_iters.'''
    n_dim = init_particles.shape[1]
    hmc_params = dict(
        step_size=step_size,
        inverse_mass_matrix=jnp.ones(n_dim),
        num_integration_steps=20,
    )
    smc = blackjax.adaptive_tempered_smc(
        logprior_fn=logprior_fn,
        loglikelihood_fn=loglik_fn,
        mcmc_step_fn=blackjax.hmc.build_kernel(),
        mcmc_init_fn=blackjax.hmc.init,
        mcmc_parameters=hmc_params,
        resampling_fn=resampling.systematic,
        target_ess=target_ess,
        num_mcmc_steps=n_mcmc_steps,
    )
    state = smc.init(init_particles)
    key = jax.random.PRNGKey(seed)

    def cond(carry):
        _, state, _ = carry
        return state.lmbda < 1.0

    def body(carry):
        i, state, key = carry
        key, subk = jax.random.split(key)
        state, _ = smc.step(subk, state)
        return i + 1, state, key

    n_iter, final_state, _ = jax.lax.while_loop(cond, body, (0, state, key))
    return final_state.particles, int(n_iter)

# Example (uncomment to run on GPU):
# init = disperse_init(2000, seed=2)
# particles, n_iter = run_tempered_smc(logL_semicoh, log_prior_raw, init)
# print("SMC tempering steps:", n_iter, "| particles:", particles.shape)


## 10. Mode clustering (study A helper)

Label resolvable modes by clustering cooled samples in a whitened metric (using a Fisher
matrix to define the scale). Lightweight, NumPy-only — no sklearn dependency.

In [ ]:
def cluster_modes(samples_raw, F_whiten=None, radius=3.0, min_count=20):
    '''Greedy distance clustering of raw-coord samples.

    samples_raw : (N, 5) pooled samples (raw coords).
    F_whiten    : (5,5) Fisher (raw coords) used to whiten distances; identity if None.
    radius      : merge threshold in whitened (sigma) units.
    Returns list of dicts: {center_raw, center_phys, count, logL_max placeholder}.
    '''
    X = np.asarray(samples_raw, dtype=float)
    if F_whiten is None:
        L = np.eye(X.shape[1])
    else:
        # whitening transform: distance^2 = dx^T F dx
        w, V = np.linalg.eigh(np.asarray(F_whiten))
        L = V @ np.diag(np.sqrt(np.clip(w, 1e-12, None))) @ V.T
    Xw = X @ L.T
    centers, members = [], []
    for i in range(Xw.shape[0]):
        placed = False
        for c in range(len(centers)):
            if np.linalg.norm(Xw[i] - centers[c]) < radius:
                members[c].append(i); placed = True
                centers[c] = Xw[np.array(members[c])].mean(0); break
        if not placed:
            centers.append(Xw[i].copy()); members.append([i])
    out = []
    for c, idx in enumerate(members):
        if len(idx) < min_count:
            continue
        cen_raw = X[np.array(idx)].mean(0)
        out.append({
            "center_raw": cen_raw,
            "center_phys": np.asarray(to_phys(jnp.asarray(cen_raw))),
            "count": len(idx),
            "frac": len(idx) / X.shape[0],
        })
    out.sort(key=lambda d: -d["count"])
    return out

# Usage after a run:
#   pooled = np.asarray(samples).reshape(-1, 5)
#   modes  = cluster_modes(pooled, F_whiten=F_fd, radius=3.0)
#   for mdl in modes: print(mdl["count"], dict(zip(PARAM_NAMES, mdl["center_phys"])))


## 11. Non-Gaussianity diagnostics (studies B & C)

Quantify how non-Gaussian the dominant mode is: Mahalanobis residual under the Fisher
Gaussian, marginal skewness / excess kurtosis, and a symmetrized Gaussian–Gaussian KL
between the sampled covariance and the Fisher prediction.

In [ ]:
from scipy import stats

def mahalanobis_residuals(samples_raw, mean_raw, Sigma):
    X = np.asarray(samples_raw, dtype=float) - np.asarray(mean_raw)
    Si = np.linalg.pinv(np.asarray(Sigma))
    return np.sqrt(np.einsum('ni,ij,nj->n', X, Si, X))

def marginal_nongauss(samples_raw):
    X = np.asarray(samples_raw, dtype=float)
    return {
        "skew":      stats.skew(X, axis=0),
        "exkurt":    stats.kurtosis(X, axis=0),   # excess (0 == Gaussian)
    }

def sym_kl_gaussian(mu0, S0, mu1, S1):
    '''Symmetrized KL between N(mu0,S0) and N(mu1,S1).'''
    mu0, mu1 = np.asarray(mu0), np.asarray(mu1)
    S0, S1 = np.asarray(S0), np.asarray(S1)
    k = mu0.size
    def kl(ma, Sa, mb, Sb):
        Sbi = np.linalg.pinv(Sb); dm = mb - ma
        (s_a, ld_a) = np.linalg.slogdet(Sa)
        (s_b, ld_b) = np.linalg.slogdet(Sb)
        return 0.5 * (np.trace(Sbi @ Sa) + dm @ Sbi @ dm - k + (ld_b - ld_a))
    return float(0.5 * (kl(mu0, S0, mu1, S1) + kl(mu1, S1, mu0, S0)))

def nongaussianity_scalar(samples_raw, Sigma_fisher, mean_raw=None):
    '''Single non-Gaussianity number per run: sym-KL(sampled || Fisher Gaussian).'''
    X = np.asarray(samples_raw, dtype=float)
    mu_s = X.mean(0) if mean_raw is None else np.asarray(mean_raw)
    S_s  = np.cov(X.T)
    return sym_kl_gaussian(mu_s, S_s, mu_s, np.asarray(Sigma_fisher))

# Usage:
#   dom = pooled[mahalanobis cut or cluster membership]
#   Sigma_fisher = np.linalg.pinv(F_fd)
#   ng = nongaussianity_scalar(dom, Sigma_fisher)


## 12. Stage 2 — likelihood on the search-statistic {f, fdot} sequence

In stage 2 the observed track is no longer taken from a known reference system but
from the **regular search pipeline** (Speri et al. / `iterative_track_search`). Its
output is a per-segment **`{f, fdot}` sequence** across the `t_alpha` bins, shape
`(Nsft,)` — exactly the shape of the stage-1 reference track. (The SVD basis is the
pipeline's internal representation of the track; the object handed off is the
reconstructed `{f, fdot}` sequence.)

So **no new likelihood is needed**: feed the search output straight into
`make_logL_track`. Everything downstream — gradient checks, Fisher/metric, sampler,
clustering, non-Gaussianity — is reused unchanged. The only modelling choice is the
per-bin uncertainty `(sigma_f, sigma_fdot)`; the search statistic's per-bin curvature
(or its det-stat weighting) is the natural place to get these.

In [ ]:
def load_search_track(search_result):
    """Adapter: pull the {f, fdot} sequence + t_alpha out of a search result.

    Replace the body with the actual accessors of your pipeline object. The
    iterative_track_search / TrackOptimizerJAX path exposes per-segment f_obs,
    fdot_obs on t_obs (== t_alpha). Returns (f_seq, fdot_seq, t_alpha).
    """
    f_seq    = np.asarray(search_result.f_obs)        # TODO: real attribute
    fdot_seq = np.asarray(search_result.fdot_obs)     # TODO: real attribute
    t_alpha  = np.asarray(search_result.t_obs)        # TODO: real attribute
    return f_seq, fdot_seq, t_alpha


def stage2_likelihood(f_seq, fdot_seq, t_alpha, mode=MODE,
                      sigma_f=1.0 / T_SFT, sigma_fdot=1.0 / T_SFT**2):
    """Build the stage-2 logL (and log-posterior) from a search {f,fdot} sequence.

    Rebuilds the track fn on the search t_alpha so model and data share a grid.
    """
    track = make_track_fn(flux_data, t_alpha, adjoint="recursive")
    logL  = make_logL_track(f_seq, fdot_seq, track=track, mode=mode,
                            sigma_f=sigma_f, sigma_fdot=sigma_fdot)
    return logL, make_log_posterior(logL), track

# Usage (stage 2):
#   f_seq, fdot_seq, t_alpha = load_search_track(search_result)
#   logL2, log_post2, track2 = stage2_likelihood(f_seq, fdot_seq, t_alpha)
#   samples, acc = run_nuts_multichain(log_post2, disperse_init(16))
print("stage-2 interface ready: search {f, fdot} sequence -> make_logL_track.")


## 13. Running on GPU & open questions

**To run the week-2 campaign on GPU**

1. Pick a grid cell `(e_f, SNR, seed)` and call `make_injection(...)` (use
   `zero_noise=True` for the noiseless reference run).
2. Rebuild `track_grad`/`track_direct` on that injection's `t_alpha` and
   `logL_semicoh = make_logL_semicoh(track_grad, data_sfts)`.
3. Global pass: `run_tempered_smc` (or dispersed `run_nuts_multichain`) → `cluster_modes`
   for the **secondary-mode census** (study A).
4. At each mode's MAP: `fisher_fd` / `fisher_gauss_newton` → `metric_report`,
   `nongaussianity_scalar` for **width/Gaussianity** (study B) and the **metric / template
   count** (study D).
5. Sweep `e_f ∈ {0,0.1,0.3,0.5,0.7}` at fixed SNR for the **headline eccentricity trend**
   (study C). `vmap`/`jit` make the 60-run grid cheap.

**Open questions (please confirm so stage 2 / the campaign are wired correctly):**

* **Sampler**: blackjax is set up here (NUTS + tempered SMC). Happy to swap in `flowMC`
  if you prefer flow-based multimodal sampling — say the word and I'll add it.
* **logL normalisation**: I used `logL = 0.5 * det_stat(...)` per the note. Confirm this
  is the intended temperature (it directly sets posterior width).
* **Stage-2 uncertainties**: the search output is a `{f, fdot}` sequence on `t_alpha`
  (confirmed). What per-bin `(sigma_f, sigma_fdot)` should weight it — flat SFT-resolution
  scales (current default), or the per-bin curvature / det-stat weighting from the search?
* **Reference point**: I fixed `(M,μ,a)=(1e6,10,0.1)`, `T_pl=0.2 yr`, matching the repo
  smoke test. Confirm or give me the reference you want for the campaign.
